In [1]:
import sys

print("Python path:", sys.executable)

Python path: C:\Users\DELL\anaconda3\envs\qafza-mlops\python.exe


# Train / Validation / Test Split

This notebook splits the labeled dataset into training, validation, and test sets before performing detailed analysis.

## 1. Load the Labeled Dataset

I load the labeled dataset created in Notebook 2.

In [2]:
import pandas as pd

# Load the labeled dataset
data = pd.read_csv("../artifacts/labeled_orders.csv")

print("Dataset shape:", data.shape)

Dataset shape: (96476, 28)


## 2. Choose the Split Strategy

A time-based split is used because delivery prediction should reflect how the model will be used in practice: training on earlier orders and evaluating on later orders.

In [3]:
# Convert purchase timestamp to datetime
data["order_purchase_timestamp"] = pd.to_datetime(
    data["order_purchase_timestamp"]
)

# Sort orders chronologically
data = data.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Check the available date range
print("Start date:", data["order_purchase_timestamp"].min())
print("End date:", data["order_purchase_timestamp"].max())

Start date: 2016-09-15 12:16:38
End date: 2018-08-29 15:00:37


## 3. Split the Dataset

The data is split chronologically into training, validation, and test sets to avoid using future orders when making decisions about the model.

In [4]:
# Define the split points
train_end = int(len(data) * 0.70)
validation_end = int(len(data) * 0.85)

# Create chronological splits
train = data.iloc[:train_end].copy()
validation = data.iloc[train_end:validation_end].copy()
test = data.iloc[validation_end:].copy()

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 67533
Validation: 14471
Test: 14472


## 4. Check Label Balance

I compare the label distribution across the training, validation, and test sets to ensure that the classes remain reasonably balanced.

In [5]:
# Compare label percentages in each split
for name, split in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(f"\n{name}:")
    print(split["delivery_label"].value_counts(normalize=True).mul(100))


Train:
delivery_label
On Time    90.971821
Late        9.028179
Name: proportion, dtype: float64

Validation:
delivery_label
On Time    94.658282
Late        5.341718
Name: proportion, dtype: float64

Test:
delivery_label
On Time    93.387231
Late        6.612769
Name: proportion, dtype: float64


## 5. Save the Data Splits

I save the training, validation, and test sets as separate artifacts for the next notebook.

In [6]:
# Save the three data splits
train.to_csv("../artifacts/train.csv", index=False)
validation.to_csv("../artifacts/validation.csv", index=False)
test.to_csv("../artifacts/test.csv", index=False)

print("Train, validation, and test files saved successfully.")

Train, validation, and test files saved successfully.


## Conclusion

The labeled dataset was split chronologically into training, validation, and test sets. This approach keeps earlier orders for training and later orders for evaluation, helping reduce the risk of using future information during model development. The three splits were saved as artifacts for the next stage.